# 06 — Fault isolation and recovery

Branch-A populations are independent of branch B. Trap and recovery counts remain separate. N, units, `thesis_evidence=false`, and descriptive-only uncertainty are explicit; missing leaves remain PENDING, never zero.


In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, pending_record
from wafer_analysis.paths import resolve_result_batch

def passed_json(batch, artifact):
    rows = []
    for path in sorted(batch.rglob(artifact)):
        status = path.parent / 'canonical-status.json'
        if status.is_file() and json.loads(status.read_text()).get('status') == 'passed':
            rows.append((path, json.loads(path.read_text())))
    return rows

rows=[]
try: batch=resolve_result_batch('e-iso-4', diagnostic_path=os.environ.get('E_ISO_4_DIR'))
except (FileNotFoundError, RuntimeError, ValueError): batch=None
containment={} if batch is None else {value.get('condition'):value for _,value in passed_json(batch,'containment.json')}
value=containment.get('infinite-loop')
if value is None:
    rows.append(pending_record('epoch recovery: infinite-loop','no passed containment.json leaf','trap and recovery events'))
else:
    recoveries=sum(int(node.get('recovery_count',0)) for node in value.get('nodes',[]))
    rows.append({'question':'epoch recovery: infinite-loop','status':'READY','condition':'infinite-loop','traps':value.get('traps_total',0),'recoveries':recoveries,'branch_a_throughput_msg_s':None,'branch_a_p95_ns':None,'units':'events','uncertainty':'descriptive only','thesis_evidence':False})
try: batch=resolve_result_batch('e-iso-7', diagnostic_path=os.environ.get('E_ISO_7_DIR'))
except (FileNotFoundError, RuntimeError, ValueError): batch=None
isolation={} if batch is None else {value.get('condition'):value for _,value in passed_json(batch,'branch-isolation.json')}
for condition in ['control','panic-attack','epoch-loop-attack']:
    value=isolation.get(condition)
    if value is None:
        row=pending_record(f'branch-A: {condition}','no passed branch-isolation.json leaf','messages/second and nanoseconds'); row['condition']=condition; rows.append(row)
    else:
        branch=value['branches']['branch_a']
        rows.append({'question':f'branch-A: {condition}','status':'READY','condition':condition,'traps':None,'recoveries':None,'branch_a_throughput_msg_s':branch['throughput']['mean_messages_per_second'],'branch_a_p95_ns':branch['latency_ns']['p95'],'units':'messages/second and nanoseconds','uncertainty':'descriptive only','thesis_evidence':False})
df=pd.DataFrame(rows); print(evidence_label(len(df[df.status=='READY']), 'events, messages/second, nanoseconds', False)); display(df)
